# Figure 1 — Fourier Truncation Rings in 1EGJ

Visualisation of Fourier truncation artefacts in the ultra-high resolution 
protein crystal structure 1EGJ (0.54 Å).

This notebook reproduces Figure 1 from:

> Alcraft, R. (2026). Title of Short Communication. 
> Acta Crystallographica Section A. doi:XXXXXXXX

Data is fetched automatically from the EBI Electron Density Server.
No local data files are required.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rae-gh/map-plane/blob/main/notebooks/Figure01_1ejg.ipynb)

In [ ]:
# Install map-plane
!pip install "map-plane @ git+https://github.com/rae-gh/map-plane.git" -q

  Cloning https://github.com/rae-gh/map-plane.git to /tmp/pip-install-7v80xoep/map-plane_05fa80d45d6b4a42816776150a04abd1
  Running command git clone --filter=blob:none --quiet https://github.com/rae-gh/map-plane.git /tmp/pip-install-7v80xoep/map-plane_05fa80d45d6b4a42816776150a04abd1
Username for 'https://github.com': ERROR: Operation cancelled by user
^C


In [ ]:
from map_plane.dmap import mapsmanager as mman
from map_plane.dmap import mapfunctions as mfun
from map_plane.dmap import mapplothelp as mph
from map_plane.vxyz import vectorthree as v3
from map_plane import MPDATA_DIR

In [ ]:
# Figure 1 — Fourier truncation rings in 1EGJ (0.54 Å)
# Plane defined by CA, C and O of Ala24, chain A
pdb_code = "1ejg"
central_atoms = ["A:24@C.A"]
linear_atoms = ["A:24@CA.A"]
planar_atoms = ["A:24@O.A"]
interpolation = "bspline"
width = 6 #Angstrom
samples = 400
# find relative path for data
mman.MapsManager().set_dir(MPDATA_DIR)
print("Data directory set to: ", MPDATA_DIR)

Data directory set to:  /home/ralcraft/.map_plane/data


In [ ]:
# Load the electron density map and protein structure from EBI
# file=1 downloads synchronously, header=1 loads map header, values=1 loads density values
ml = mman.MapsManager().get_or_create(pdb_code, file=1, header=1, values=1)

# Create the map functions object for this structure
mf = mfun.MapFunctions(pdb_code, ml.mobj, ml.pobj, interpolation)

# Define the plane through three atomic coordinates
slice_vectors = []
for i in range(len(central_atoms)):
    central_atom = central_atoms[i]  # central_atom: origin of the plane
    linear_atom = linear_atoms[i]    # linear_atom:  defines the x-axis direction
    planar_atom = planar_atoms[i]    # planar_atom:  defines the plane orientation

    # Retrieve atomic coordinates and convert to 3D vectors
    cc = v3.VectorThree().from_coords(ml.pobj.get_coords_key(central_atom))
    ll = v3.VectorThree().from_coords(ml.pobj.get_coords_key(linear_atom))
    pp = v3.VectorThree().from_coords(ml.pobj.get_coords_key(planar_atom))
    slice_vectors.append((cc, ll, pp))

In [ ]:
filename = "SHOW"  # "SHOW" displays inline rather than saving to file

for cc, ll, pp in slice_vectors:
    # Sample electron density on a 2D grid in the defined plane
    # deriv=0 returns full density (deriv=1 would return first derivative)
    vals2d = mf.get_slice(cc, ll, pp, width, samples, interpolation, deriv=0, ret_type="2d")

    # Plot as a heatmap
    # min_percent and max_percent clip the density range to enhance contrast
    # hue="WB" uses white-to-black colour scheme
    mplot = mph.MapPlotHelp(filename)
    mplot.make_plot_slice_2d(vals2d,
                             min_percent=1,      # clip bottom 1% of density values
                             max_percent=0.15,   # clip top 0.15% to reveal ring detail
                             samples=samples,
                             width=width,
                             title=pdb_code,
                             plot_type="heatmap",
                             hue="WB")